<a href="https://colab.research.google.com/github/yourusername/banking-customer-segmentation/blob/main/customer_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Banking Customer Segmentation Analysis

This notebook demonstrates comprehensive customer segmentation techniques for banking customers using machine learning approaches.

## Learning Objectives
- Understand RFM (Recency, Frequency, Monetary) analysis
- Apply K-means clustering for customer segmentation
- Generate actionable business insights
- Create compelling visualizations for stakeholder communication

In [ ]:
# Install required packages
! pip install pandas numpy matplotlib seaborn scikit-learn plotly

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Load and Explore Data

First, let's load our synthetic banking data and explore its structure.

In [ ]:
# Load data (make sure you have run the data generator first)
try:
    data = pd.read_csv('synthetic_bank_data.csv')
    print(f"Data loaded successfully! Shape: {data.shape}")
except FileNotFoundError:
    print("Error: synthetic_bank_data.csv not found.")
    print("Please run the data generator notebook first to create the dataset.")
    
# Display basic information
data.head()

In [ ]:
# Explore data structure
print("Dataset Information:")
print(data.info())
print("\nSummary Statistics:")
data.describe()

## 2. RFM Analysis

RFM Analysis segments customers based on:
- **Recency**: How recently did the customer transact?
- **Frequency**: How often does the customer transact?
- **Monetary**: How much does the customer spend/maintain as balance?

In [ ]:
# [Include the RFM analysis code from the CustomerSegmentation class]
# This would include the RFM scoring and segmentation logic

## 3. Data Preprocessing for Clustering

In [ ]:
# Select features for clustering
clustering_features = [
    'age', 'annual_income', 'account_balance', 'credit_score',
    'monthly_transactions', 'num_products', 'tenure_years',
    'recency_score', 'frequency_score', 'monetary_score'
]

# Prepare clustering data
cluster_data = data[clustering_features].copy()
cluster_data = cluster_data.fillna(cluster_data.median())

# Scale the features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cluster_data)

print(f"Prepared {scaled_data.shape[0]} samples with {scaled_data.shape[1]} features for clustering")

## 4. Finding Optimal Number of Clusters

In [ ]:
# Determine optimal number of clusters using elbow method and silhouette analysis
max_clusters = 10
cluster_range = range(2, max_clusters + 1)
inertias = []
silhouette_scores = []

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(scaled_data)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(scaled_data, cluster_labels))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Elbow curve
axes[0].plot(cluster_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method For Optimal k')
axes[0].grid(True)

# Silhouette scores
axes[1].plot(cluster_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Find optimal clusters
optimal_clusters = cluster_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters: {optimal_clusters}")

## 5. K-Means Clustering

In [ ]:
# Perform K-means clustering with optimal number of clusters
n_clusters = optimal_clusters  # You can modify this if needed

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_data)

# Add cluster labels to the data
data_clustered = cluster_data.copy()
data_clustered['Cluster'] = cluster_labels

print(f"Clustering completed with {n_clusters} clusters")
print(f"Silhouette score: {silhouette_score(scaled_data, cluster_labels):.3f}")
print("\nCluster distribution:")
print(data_clustered['Cluster'].value_counts().sort_index())

## 6. Cluster Analysis and Profiling

In [ ]:
# Analyze cluster characteristics
cluster_summary = data_clustered.groupby('Cluster').agg({
    'age': 'mean',
    'annual_income': 'mean',
    'account_balance': 'mean',
    'credit_score': 'mean',
    'monthly_transactions': 'mean',
    'num_products': 'mean',
    'tenure_years': 'mean'
}).round(2)

print("Cluster Characteristics:")
print(cluster_summary)

# Add cluster sizes
cluster_sizes = data_clustered['Cluster'].value_counts().sort_index()
cluster_summary['Size'] = cluster_sizes
cluster_summary['Percentage'] = (cluster_sizes / len(data_clustered) * 100).round(1)

print("\nCluster Summary with Sizes:")
print(cluster_summary)

## 7. Cluster Visualization

In [ ]:
# Create comprehensive cluster visualizations
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. PCA visualization
pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(scaled_data)

scatter = axes[0, 0].scatter(pca_data[:, 0], pca_data[:, 1], 
                           c=cluster_labels, cmap='viridis', alpha=0.6)
axes[0, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
axes[0, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
axes[0, 0].set_title('Customer Clusters (PCA View)')
plt.colorbar(scatter, ax=axes[0, 0])

# 2. Income vs Balance
scatter = axes[0, 1].scatter(data_clustered['annual_income'], data_clustered['account_balance'],
                           c=data_clustered['Cluster'], cmap='viridis', alpha=0.6)
axes[0, 1].set_xlabel('Annual Income ($)')
axes[0, 1].set_ylabel('Account Balance ($)')
axes[0, 1].set_title('Income vs Balance by Cluster')
plt.colorbar(scatter, ax=axes[0, 1])

# 3. Age vs Credit Score
scatter = axes[0, 2].scatter(data_clustered['age'], data_clustered['credit_score'],
                           c=data_clustered['Cluster'], cmap='viridis', alpha=0.6)
axes[0, 2].set_xlabel('Age')
axes[0, 2].set_ylabel('Credit Score')
axes[0, 2].set_title('Age vs Credit Score by Cluster')
plt.colorbar(scatter, ax=axes[0, 2])

# 4. Cluster size distribution
cluster_counts = data_clustered['Cluster'].value_counts().sort_index()
axes[1, 0].bar(cluster_counts.index, cluster_counts.values, color='skyblue')
axes[1, 0].set_xlabel('Cluster')
axes[1, 0].set_ylabel('Number of Customers')
axes[1, 0].set_title('Cluster Size Distribution')

# 5. Average income by cluster
avg_income = data_clustered.groupby('Cluster')['annual_income'].mean()
axes[1, 1].bar(avg_income.index, avg_income.values, color='lightgreen')
axes[1, 1].set_xlabel('Cluster')
axes[1, 1].set_ylabel('Average Income ($)')
axes[1, 1].set_title('Average Income by Cluster')

# 6. Products vs Transactions
scatter = axes[1, 2].scatter(data_clustered['num_products'], data_clustered['monthly_transactions'],
                           c=data_clustered['Cluster'], cmap='viridis', alpha=0.6)
axes[1, 2].set_xlabel('Number of Products')
axes[1, 2].set_ylabel('Monthly Transactions')
axes[1, 2].set_title('Products vs Transactions by Cluster')
plt.colorbar(scatter, ax=axes[1, 2])

plt.tight_layout()
plt.show()

## 8. Business Insights and Segment Labeling

In [ ]:
# Define business segments based on cluster characteristics
def assign_business_labels(cluster_summary):
    labels = {}
    strategies = {}
    
    for cluster in cluster_summary.index:
        row = cluster_summary.loc[cluster]
        
        if row['annual_income'] > 80000 and row['account_balance'] > 50000:
            labels[cluster] = 'High-Value Customers'
            strategies[cluster] = 'Premium services, wealth management, exclusive offers'
            
        elif row['age'] < 35 and row['annual_income'] < 50000:
            labels[cluster] = 'Young Professionals'
            strategies[cluster] = 'Digital banking, career-building products, competitive rates'
            
        elif row['tenure_years'] > 15:
            labels[cluster] = 'Long-term Loyal'
            strategies[cluster] = 'Retention programs, loyalty rewards, referral incentives'
            
        elif row['monthly_transactions'] < 5:
            labels[cluster] = 'Low-Engagement'
            strategies[cluster] = 'Activation campaigns, simplified products, education'
            
        else:
            labels[cluster] = 'Standard Customers'
            strategies[cluster] = 'Cross-selling, product bundling, satisfaction surveys'
    
    return labels, strategies

business_labels, strategies = assign_business_labels(cluster_summary)

# Display business insights
print("CUSTOMER SEGMENT BUSINESS INSIGHTS")
print("="*60)

for cluster in sorted(business_labels.keys()):
    row = cluster_summary.loc[cluster]
    print(f"\nSEGMENT {cluster}: {business_labels[cluster].upper()}")
    print("-" * 50)
    print(f"Size: {row['Size']:,} customers ({row['Percentage']:.1f}%)")
    print(f"Average Income: ${row['annual_income']:,.0f}")
    print(f"Average Balance: ${row['account_balance']:,.0f}")
    print(f"Average Age: {row['age']:.1f} years")
    print(f"Average Products: {row['num_products']:.1f}")
    print(f"Strategy: {strategies[cluster]}")
    
    # Calculate revenue opportunity
    revenue_opportunity = (row['account_balance'] * 0.02 + row['num_products'] * 50) * row['Size']
    print(f"Revenue Opportunity: ${revenue_opportunity:,.0f} annually")

## 9. Interactive Dashboard (Optional)

In [ ]:
# Create an interactive dashboard using plotly (if available)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Interactive scatter plot
    fig = px.scatter(data_clustered, 
                     x='annual_income', 
                     y='account_balance',
                     color='Cluster',
                     size='monthly_transactions',
                     hover_data=['age', 'credit_score', 'num_products'],
                     title='Interactive Customer Segments',
                     labels={'annual_income': 'Annual Income ($)',
                            'account_balance': 'Account Balance ($)'})
    
    fig.show()
    
except ImportError:
    print("Plotly not available. Install with: pip install plotly")

## 10. Export Results

In [ ]:
# Add segment labels to original data
data_export = data.copy()
data_export['Cluster'] = cluster_labels
data_export['Segment_Label'] = data_export['Cluster'].map(business_labels)
data_export['Strategy'] = data_export['Cluster'].map(strategies)

# Export to CSV
data_export.to_csv('customer_segments.csv', index=False)
print("Results exported to 'customer_segments.csv'")

# Display final summary
print("\nFINAL SEGMENTATION SUMMARY:")
print(data_export['Segment_Label'].value_counts())

## Student Exercises

### Beginner Level
1. **Parameter Tuning**: Try different numbers of clusters (3, 4, 6, 8) and compare results
2. **Feature Selection**: Remove some features and observe how clustering changes
3. **Visualization**: Create additional plots to explore cluster characteristics

### Intermediate Level
4. **Alternative Algorithms**: Try hierarchical clustering or DBSCAN
5. **Feature Engineering**: Create new features like income-to-balance ratio and see impact
6. **Business Rules**: Modify the business labeling rules based on your understanding

### Advanced Level
7. **Model Validation**: Implement cluster stability analysis
8. **Predictive Modeling**: Build models to predict which cluster a new customer belongs to
9. **A/B Testing Framework**: Design experiments to test segment-specific strategies